# COMP9321 26T3 Assignment 2 v1.0.0 – (20 Marks)
## Due Date: 2026-10-29 - 18:00 
## Overview

In this assignment, you will build a RESTful API using **Flask-RESTX** that supports two types of accounts — **Admin** and **User** — interacting with airports, airlines and routes data.

- The **Admin** can create and manage User accounts and import airline operations data from CSV files.
- **Users** can browse, search and explore airports, airlines, and routes.
- **Users** can create and manage their own private travel plans.
- The **Admin** can also act as a **User**.

Your API must:

- Store data in a local **SQLite** database.
- Expose endpoints that are fully documented and testable via **Swagger UI**.

> The three CSV files provided with this assignment contain a small synthetic dataset created for teaching purposes. Although the schema is inspired by public aviation datasets, the records are not real and should not be used for any other purpose.

---

## Development Environment

You must submit **two Python files**:

- `z[your_zid]_api.py`: Your Flask-RESTX API implementation.
- `z[your_zid]_tests.py`: Your automated test suite (REST client).

### Technical Requirements

- Use the provided `requirements.txt` to set up your virtual environment.
- Use only the libraries available in the environment (e.g., `pandas`, `matplotlib`).
- Use **Python 3.13 only**.
- Use only **SQLite** as your database. The database file must:
  - Be created automatically at runtime if it does not exist. If the database exists, reuse it.
  - Not use absolute paths (e.g., `C://A/B/.../mydb`).
  - Not be submitted.
  - Persist all the resources (e.g., accounts, airports, airlines, routes, travel plans).
- You will be penalised up to 50% of the mark if you fail to persist the data properly in your SQLite database (for example, if data is held only in memory, or the database is recreated from scratch on every restart).
- You can structure your database models in any way that best suits your data and application requirements, as long as the design is logical, consistent, and well-justified.
---

## Requirement Sets

Your API must follow **RESTful design best practices**, including but not limited to:

- Properly designing and grouping endpoints.
- Using appropriate HTTP methods.
- Avoiding redundant or excessive endpoints.
- Avoiding unnecessarily large payloads.
- Using appropriate parameter types (e.g., Query Parameters, Header Fields, Path Parameters, JSON Payload).
- Validating user input and handling errors properly.
- Considering backward compatibility — how future changes may affect existing users.
- Providing efficient ways to explore data.

> **API design is assessed.** For every requirement set, marks are awarded not only for working functionality but also for the quality of your RESTful design and Swagger documentation. Where this specification leaves a design choice open, you are expected to make an appropriate decision and justify it in your Swagger documentation.

You must generate a **fully documented Swagger UI** using Flask-RESTX.

Each endpoint must include:

- Summary and description
- Request and response models
- Status codes and error messages
- Role-based access notes

The Swagger UI must:

- Be accessible via a browser at: `http://127.0.0.1:5000`
- Be fully interactive
- Be self-explanatory
- Be synchronised with your implementation

> Failure to implement a working Swagger UI will result in loss of marks for all other sections.

Design one or more endpoints per requirement set based on the following requirements.

---

### Requirement Set 1: User Management and Access Control (5 marks)

The API must support two account types: **Admin** and **User**.

- There is only one dedicated **Admin** account with username `admin`. There is no limit to the number of **User** accounts.
- The `admin` account can access every endpoint. However, resources that belong to an account (e.g., travel plans in Requirement Set 4) remain private to their owner: the `admin` can only view and manage the travel plans that the `admin` account itself created.
- The `admin` can create and delete User accounts.
- The `admin` can activate or deactivate User accounts at any time. API access is available only to active accounts. Deactivated accounts lose their access to the API unless they are reactivated.
- The `admin` account itself cannot be deleted or deactivated.
- The `admin` can retrieve a list of all accounts and their details. Credentials (passwords or password hashes) must never be returned by the API.

Your system must automatically create the following accounts on first run:

| Username | Password | Role  |
|----------|----------|-------|
| admin    | admin    | Admin |
| user     | user     | User  |

Additional constraints:

- The Admin can perform all actions available in the system. As such, the Admin can act as a normal User as well.
- Users can perform all actions **except**:
  - User management
  - Importing airports, airlines, routes, etc.
- You must choose, implement and justify an appropriate authentication and authorisation mechanism, and document it in your Swagger UI. Requests from unauthenticated, deactivated, or insufficiently privileged accounts must be rejected with appropriate status codes and error messages.

---

### Requirement Set 2: Importing Airports, Airlines, and Routes (2 marks)

Only the `admin` account can import airports, airlines, and routes data.

- The Admin must be able to import the three CSV files via the API. The files must be uploaded as file attachments using the `multipart/form-data` Content-Type.
  - [airports.csv](`airports.csv`)
  - [airlines.csv](`airlines.csv`)
  - [routes.csv](`routes.csv`)
- The imported data must be stored in the local SQLite database.
- You must process the provided CSV files and extract any information required to support the remaining requirement sets.

### Data Import Requirements

- **Independent dataset imports:** The Admin must be able to import any subset of the three CSV files independently (e.g., only `airports.csv`, or `airports.csv` and `airlines.csv` together). Each dataset must be importable without requiring the other files to be included in the same request.

- **Replace existing data:** Importing a CSV file must replace all existing records for the corresponding dataset with the records in the uploaded file. Datasets not included in the import request must remain unchanged.

- **File validation:** The provided sample files define the expected schema of each dataset. The API must validate every uploaded file against its expected schema before making any database changes. At a minimum, validation must cover the column names, the data type of each column, the uniqueness of identifiers, and the consistency of related fields (e.g., a route's `origin_airport_id` and `origin_iata` must refer to the same airport). If a file is invalid, malformed, or does not conform to the expected schema, the API must return a clear and appropriate error response.

- **Referential integrity:** An import must preserve referential integrity across the three datasets. If, after applying all the files in the request, any route would reference an airline or airport that does not exist, the API must reject the entire import request and return an appropriate error response, such as `409 Conflict`. Note that when several files are included in one request, integrity must be checked against the combined result of the request, not against the data currently in the database.

- **Atomicity:** Each import request must be processed atomically. If validation fails or the import cannot be completed while preserving referential integrity, the request must be rejected without changing any database records.

- **Existing travel plans:** Replacing the routes dataset may remove routes that are referenced by existing travel plans (Requirement Set 4). Your API must handle this situation in a well-defined way (for example, by rejecting the import, or by accepting it and marking the affected plans), and your choice must be documented in the Swagger UI.

---

### Requirement Set 3: Exploring Airports, Airlines and Routes (3 marks)

All accounts must be able to explore the airline operations data through appropriate RESTful endpoints.

Your API must provide endpoints that allow users to retrieve detailed information about:

- Airports
- Airlines
- Routes

You should also consider the relationships between these resources and provide appropriate ways to retrieve related objects. For example, users should be able to perform operations including but not limited to:

- Retrieve the airlines operating from a particular airport.
- Retrieve the routes associated with a particular airline.
- Retrieve routes departing from a particular airport.

You may also provide search functionality where appropriate. If you implement text-based search, it should support case-insensitive and partial matching, and you should consider whether fuzzy matching is appropriate for the resource being searched.

The API should provide sufficient information for a client to navigate between related resources without requiring knowledge of the underlying SQLite database structure.

---

### Requirement Set 4: My Travel Plans (3 marks)

A travel plan represents a collection of routes that a user is interested in, together with some metadata.

- All accounts must be able to create private travel plans.
- Different accounts may have travel plans with the same name.
- There is no limit to the number of travel plans per account.
- At a minimum, a travel plan has:
    - A name
    - A travel departure date
    - A travel return date (which must not be earlier than the departure date)
    - The set of routes it contains
- Your API must allow a user to create, retrieve, modify and remove their travel plans, and to add routes to and remove routes from an existing plan.

How you structure these resources — for example, how routes are referenced, whether routes within a plan are exposed as a nested resource, and how a consumer navigates from a plan to the full details of its routes, airlines and airports — is a design decision that will be assessed. There are no additional marks for adding extra fields; the marks are for a design that lets API consumers access the underlying resources in an appropriate and flexible way.

> Travel plans must be private and accessible only to their owner. This applies to the `admin` account as well (see Requirement Set 1).

---

### Requirement Set 5: Exporting Travel Plans (4 marks)

The purpose of this requirement is to let an account export **its own travel plans**.

Your output must be presented in the following formats based on the user's request:
  - As a downloadable JSON file containing all of the requesting account's travel plans, ordered by departure date. The file must be self-contained: each plan must include the details of its routes (and the related airlines and airports), not only their identifiers.
  - As a visualisation image (e.g., generated with `matplotlib`) that can be viewed in a browser, showing up to the five *future* travel plans with the nearest departure dates. A plan is considered a future plan if its departure date is later than the current date. The content of the visualisation is your design decision (for example, a timeline of the plans' travel dates, or a map of the planned routes), but it must be meaningful to the user. If the account has no future travel plans, the API must respond appropriately.

How the client selects the output format (e.g., content negotiation or a query parameter) is a design decision that will be assessed.

---

### Requirement Set 6: Testing (3 marks)

You must also submit: `z[your_zid]_tests.py`

Your API is as good as your test cases. Your automated test suite must include:

- Tests covering the main functionality of each requirement set
- Edge case tests

Each test must clearly document:

- The requirement number
- The purpose of the test

Your tests must include assertions for:

- HTTP status codes
- JSON response structure and data correctness
- Error messages for invalid or unauthorised cases

Your tests must be written as a REST client (e.g., using the `requests` library) and must be runnable with `pytest z[your_zid]_tests.py` while your API is running at `http://127.0.0.1:5000` with a freshly created database and the three CSV files located in the working directory. Tests must not depend on the order in which they are executed within the file beyond what your own setup code establishes.

---

*~ end of assignment specification ~*